## Using pandas for data analysis
- Pandas loads the entire CSV into a DataFrame in one line and handles headers

In [4]:
import pandas as pd

class CsvReader:
    def __init__(self, file_path):
        self.file_path = file_path

    def read_with_pandas(self):
        # Reads the CSV and returns a pandas DataFrame
        df = pd.read_dataframe(self.file_path) # type: ignore (Conceptual note: pd.read_csv is standard)
        df = pd.read_csv(self.file_path)
        return df

# Usage
# reader = CsvReader("data.csv")
# my_data = reader.read_with_pandas()

ModuleNotFoundError: No module named 'pandas'

## Using the Built-n CSV Module 
- The csv.DictReader parses each row into a dictionary where keys are column headers
- This option would work better for light operations

In [ ]:
import csv

class CsvReader:
    def __init__(self, file_path):
        self.file_path = file_path

    def read_with_csv_module(self):
        # Opens file safely and reads rows as dictionaries
        with open(self.file_path, mode='r', encoding='utf-8') as file:
            csv_reader = csv.DictReader(file)
            return list(csv_reader)

# Usage
# reader = CsvReader("data.csv")
# my_data = reader.read_with_csv_module()

## Microservice Architechture 
- Run spring boot as the core backend and offload data shipping/processing to a python api to keep both systems efficient

In [ ]:
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

class FileRequest(BaseModel):
    file_path: str

class CsvReader:
    def __init__(self, file_path: str):
        self.file_path = file_path
    
    def read_with_pandas(self) -> pd.DataFrame:
        # pd.read_csv is the correct method to load a CSV file
        return pd.read_csv(self.file_path)

@app.post("/read-csv")
def read_csv_endpoint(request: FileRequest):
    try:
        reader = CsvReader(request.file_path)
        df = reader.read_with_pandas()
        
        # Convert DataFrame to a dictionary/JSON structure for network shipping
        return {"status": "success", "data": df.to_dict(orient="records")}
    except FileNotFoundError:
        raise HTTPException(status_code=404, detail="The specified CSV file was not found.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# run this python app using Uvicorn: uvicorn main:app --reload --port 8000

## Build the Java Client (Spring Boot)
- Create this service in your spring booth application to fetch he processed data from python over the network

In [ ]:

package com.carauction.data.service;

import org.springframework.stereotype.Service;
import org.springframework.web.client.RestTemplate;
import java.util.HashMap;
import java.util.Map;
import java.util.List;

@Service
public class DataShippingService {

    private final RestTemplate restTemplate = new RestTemplate();
    private final String PYTHON_API_URL = "http://localhost:8000/read-csv";

    public List<Map<String, Object>> fetchCsvData(String csvFilePath) {
        // Prepare the payload matching the Python Pydantic schema
        Map<String, String> request = new HashMap<>();
        request.put("file_path", csvFilePath);

        // Make the network call to the Python service
        Map<String, Object> response = restTemplate.postForObject(PYTHON_API_URL, request, Map.class);
        
        if (response != null && "success".equals(response.get("status"))) {
            return (List<Map<String, Object>>) response.get("data");
        }
        
        throw new RuntimeException("Failed to read CSV data from Python API");
    }
}
